## Dorado Spillway-Release Workflow (Unstructured Delft3DFM)

This notebook mirrors your spillway-particle workflow and is adapted to the current Lake_Pontchartrain_Model project layout.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import time as tm
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from pyproj import Proj

import dorado
import dorado.particle_track as pt

starttm = tm.time()

### 1. Configure Paths and Runtime Settings

In [ ]:
PROJECT_ROOT = Path.cwd().parent
NOTEBOOK_DATA = PROJECT_ROOT / 'notebooks' / 'data'

scenario_name = 'run_02_spillway'
model_run = '2013_01'

unstructured_path = PROJECT_ROOT / 'outputs' / model_run / f'unstructured_model_Delft3DFM_{model_run}.txt'
boundaries_path = NOTEBOOK_DATA / 'boundaries_latlon.xlsx'

path2folder = PROJECT_ROOT / 'outputs' / scenario_name
path2folder.mkdir(parents=True, exist_ok=True)
(path2folder / 'figs').mkdir(parents=True, exist_ok=True)

# Time-window settings (hours on model timeline)
startRange = 1438
endRange = 8015
hydrodt = 3
timesteps = (endRange - startRange) // hydrodt
model_timestep = 3600 * hydrodt

print('Output folder:', path2folder)
print('Timesteps to run:', timesteps)

### 2. Load Unstructured Inputs and Boundaries

In [ ]:
if not unstructured_path.exists():
    raise FileNotFoundError(f'Missing unstructured file: {unstructured_path}')
if not boundaries_path.exists():
    raise FileNotFoundError(f'Missing boundaries file: {boundaries_path}')

with open(unstructured_path, 'r') as f:
    unstructured = json.load(f)
boundaries = pd.read_excel(boundaries_path)

print('Loaded unstructured:', unstructured_path)
print('Loaded boundaries:', boundaries_path)

### 3. Build Particle-Release Schedule

Use CSV daily totals if available; otherwise fallback to a parabolic release profile.

In [ ]:
CSV_PATH = NOTEBOOK_DATA / 'bcs_2020_particles_only_2.csv'
DATE_COL = 'date'
VAL_COL = 'Q_particles_scaled'
steps_per_day = int(round(24 / hydrodt))

if CSV_PATH.exists():
    daily = pd.read_csv(CSV_PATH)
    daily[DATE_COL] = pd.to_datetime(daily[DATE_COL])
    daily = daily.sort_values(DATE_COL)
    daily[VAL_COL] = pd.to_numeric(daily[VAL_COL], errors='coerce').fillna(0)
    daily = daily.groupby(DATE_COL, as_index=False)[VAL_COL].sum()

    thr = 0
    open_mask = daily[VAL_COL] > thr
    if not open_mask.any():
        raise ValueError('No spillway-open days found (all daily totals <= threshold).')

    start_date = daily.loc[open_mask, DATE_COL].min()
    end_date = daily.loc[open_mask, DATE_COL].max()

    daily = (daily.set_index(DATE_COL)
                .reindex(pd.date_range(start_date, end_date, freq='D'))
                .fillna({VAL_COL: 0})
                .rename_axis(DATE_COL)
                .reset_index())

    daily_totals = np.round(daily[VAL_COL].to_numpy()).astype(int)
    daily_totals[daily_totals < 0] = 0

    particles_list = []
    for N_day in daily_totals:
        base = N_day // steps_per_day
        rem = N_day % steps_per_day
        day_steps = np.full(steps_per_day, base, dtype=int)
        if rem > 0:
            day_steps[:rem] += 1
        particles_list.append(day_steps)

    particles = np.concatenate(particles_list)
    print('Particle mode: CSV-driven')
    print('Total particles:', int(particles.sum()))
else:
    total_particles = 20112
    total_timesteps_spillway = int(114 * (24 / hydrodt))
    time_array = np.linspace(0, 1, total_timesteps_spillway)
    parabolic_distribution = 4 * time_array * (1 - time_array)
    min_particles = 1
    scaled_distribution = (parabolic_distribution / np.sum(parabolic_distribution)) * (total_particles - min_particles * total_timesteps_spillway)
    particles = np.round(scaled_distribution + min_particles).astype(int)
    print('Particle mode: fallback parabolic (CSV not found)')
    print('Expected CSV location:', CSV_PATH)
    print('Total particles:', int(particles.sum()))

plt.figure(figsize=(6, 4), dpi=300)
plt.plot(particles, marker='o', markersize=2, linewidth=0.8)
plt.title('Particles Per Timestep')
plt.xlabel('Time Step')
plt.ylabel('Particle Count')
plt.show()

### 4. Project Coordinates, Grid Data, and Load Cell Types

In [ ]:
utm_proj = Proj(proj='utm', zone=15, ellps='WGS84')
(unstructured['x'], unstructured['y']) = utm_proj(unstructured['x_face'], unstructured['y_face'])
unstructured['x'] = np.round(unstructured['x'], 2)
unstructured['y'] = np.round(unstructured['y'], 2)
boundaries['x'], boundaries['y'] = utm_proj(boundaries['x'], boundaries['y'])

coordinates = [(round(float(x), 2), round(float(y), 2)) for x, y in zip(unstructured['x'], unstructured['y'])]
resolution = 100
knn = 5
myInterp, elevation = pt.unstruct2grid(coordinates, unstructured['elevation'], resolution, knn, boundary=boundaries, crop=True)

# Optional patch for known cropped edge in this setup.
interpolate_indices = (320, 324, 779, 782)
sub = elevation[interpolate_indices[0]:interpolate_indices[1], interpolate_indices[2]:interpolate_indices[3]]
elevation[interpolate_indices[0]:interpolate_indices[1], interpolate_indices[2]:interpolate_indices[3]] = pd.DataFrame(sub).interpolate().values

celltype_initial_path = NOTEBOOK_DATA / 'gridcell_type.csv'
roi_candidates = [
    NOTEBOOK_DATA / 'gridcell_type_lp_isolated.csv',
    NOTEBOOK_DATA / 'gridcelltype_lpisolated.csv',
    NOTEBOOK_DATA / 'cell_type_all_100m_roi.csv'
]
roi_path = next((p for p in roi_candidates if p.exists()), None)
if roi_path is None:
    raise FileNotFoundError('Could not find ROI gridcell file in notebooks/data.')

celltype_initial = np.genfromtxt(celltype_initial_path, delimiter=',')
gridcelltype_lpisolated = np.genfromtxt(roi_path, delimiter=',')
gridcelltype_lpisolated[np.isnan(gridcelltype_lpisolated)] = 2

print('Loaded celltype_initial:', celltype_initial_path)
print('Loaded gridcelltype_lpisolated:', roi_path)

### 5. Define Seed Regions and Initial Release Location

In [ ]:
regions = np.zeros_like(elevation, dtype='int')
seed_locations = np.argwhere(gridcelltype_lpisolated == 1)
regions[gridcelltype_lpisolated == 1] = 1

# Example fixed release line (adjust as needed)
seed_yloc = list(range(180, 190))
seed_xloc = [400] * len(seed_yloc)

plt.figure(figsize=(6, 4), dpi=300)
plt.scatter(seed_yloc, seed_xloc, c='r', s=5)
plt.imshow(elevation, cmap='terrain', origin='lower', vmin=0, vmax=-25)
plt.colorbar(fraction=0.03)
plt.title('Check Particle Release Location - Red Dots')
plt.gca().invert_yaxis()
plt.show()

### 6. Run Dorado Iterations

In [ ]:
params = pt.modelParams()
params.theta = 1.0
params.gamma = 0.05
params.diff_coeff = 0.8
params.cell_type = np.where(np.isnan(elevation), 2, celltype_initial)
params.dx = resolution
params.dry_depth = 0.01
params.verbose = False

timesteps_to_run = min(timesteps, len(unstructured['timesteps']) - startRange, len(particles))
target_times = np.arange(0, model_timestep * (timesteps_to_run + 1), model_timestep)
target_times = [int(t) for t in target_times]
prev_counts = None

print('Actual timesteps run:', timesteps_to_run)

for i in range(timesteps_to_run):
    tti = startRange + i * hydrodt

    stage = myInterp(unstructured['timesteps'][tti][0])
    depth = myInterp(unstructured['timesteps'][tti][1])
    u = myInterp(unstructured['timesteps'][tti][2])
    v = myInterp(unstructured['timesteps'][tti][3])

    for arr in [stage, depth, u, v]:
        sub = arr[interpolate_indices[0]:interpolate_indices[1], interpolate_indices[2]:interpolate_indices[3]]
        arr[interpolate_indices[0]:interpolate_indices[1], interpolate_indices[2]:interpolate_indices[3]] = pd.DataFrame(sub).interpolate().values

    params.topography = elevation
    params.stage = stage
    params.qx = u * depth
    params.qy = v * depth
    params.u = u
    params.y = v

    particle = pt.Particles(params)
    if i == 0:
        particle.generate_particles(int(particles[i]), seed_xloc, seed_yloc, seed_time=0, method='exact')
    else:
        particle.generate_particles(int(particles[i]), seed_xloc, seed_yloc, seed_time=float(target_times[i]), method='exact', previous_walk_data=walk_data)

    walk_data = particle.run_iteration(target_times[i])
    xi, yi, ti = dorado.routines.get_state(walk_data)

    x_key = 'xinds' if 'xinds' in walk_data else 'x_inds'
    t_key = 'travel_times'
    x_lists = walk_data[x_key]
    t_lists = walk_data.get(t_key, [[] for _ in x_lists])
    walk_data[t_key] = t_lists

    if prev_counts is None or len(prev_counts) != len(x_lists):
        prev_counts = [0] * len(x_lists)

    t_start = target_times[i - 1] if i > 0 else 0
    t_end = target_times[i]

    for p, x_list in enumerate(x_lists):
        cur_len = len(x_list)
        added = cur_len - prev_counts[p]
        if added <= 0:
            continue
        last_time = t_lists[p][prev_counts[p] - 1] if prev_counts[p] > 0 else t_start
        seg = np.linspace(last_time, t_end, added + 1, endpoint=True)[1:]
        seg = [int(s) if s.is_integer() else float(s) for s in seg]
        if len(t_lists[p]) < cur_len:
            t_lists[p].extend([None] * (cur_len - len(t_lists[p])))
        t_lists[p][prev_counts[p]:cur_len] = seg
        prev_counts[p] = cur_len

    if i % 24 == 0 or i == timesteps_to_run - 1:
        fig, ax = plt.subplots(figsize=(6, 4), dpi=200)
        ax.scatter(yi, xi, c='firebrick', s=1)
        im = ax.imshow(particle.depth * -1, vmin=0, vmax=-25)
        plt.imshow(regions, cmap='bone', alpha=0.3)
        plt.title(f'Time: {int((target_times[i] / 3600) // 24)} Days, {int((target_times[i] / 3600) % 24)} Hours')
        cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label('Depth (m)')
        ax.set_facecolor('whitesmoke')
        plt.savefig(path2folder / f'output_by_dt{i}.png')
        plt.close()

print('Routing complete.')

### 7. Exposure Time and Save Outputs

In [ ]:
exposure_times = pt.exposure_time(walk_data, regions)
dorado.routines.plot_exposure_time(
    walk_data,
    exposure_times,
    str(path2folder / 'figs'),
    timedelta=86400,
    nbins=20
)

json.dump(walk_data, open(path2folder / f'{scenario_name}_WalkData_From_Spillway.txt', 'w'))
np.savez_compressed(
    path2folder / f'{scenario_name}_Exposure_From_Spillway.npz',
    exposure_times=exposure_times,
    elevation=elevation,
    gridcelltype_lpisolated=gridcelltype_lpisolated,
    model_timestep=model_timestep
)

elapsed = (tm.time() - starttm) / 60
hrs, mins = divmod(elapsed, 60)
print(f'Time Taken: {int(hrs)} hours, {round(mins)} minutes')